[![Homepage](https://img.shields.io/badge/homepage-blueviolet?logo=htmx)](https://www.bendai.org/CUHK-STAT3009/)
[![GitHub](https://img.shields.io/badge/GitHub-black.svg?logo=github)](https://github.com/statmlben/CUHK-STAT3009)
[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/statmlben/CUHK-STAT3009/blob/main/HW/STAT3009_HW1.ipynb)

# STAT3009 Homework 1 — Read, extend, and evaluate a recommender

**Coverage:** Machine Learning I  
**Total:** 100 points  
**Due:** Thursday, 8 October 2026, before lecture

This homework follows one focused workflow:

```text
prepared rating data
    → inspect the train/test relationship
    → extend the mean baselines once
    → evaluate and explain the result
```

The setup code is provided. You do **not** need to rewrite the ID encoding or the three classroom baselines. Each question includes a short coding task followed by interpretation.

## Learning goals

By the end, you should be able to:

1. explain the input–target split for rating prediction;
2. use pandas, sets, loops, and NumPy masks on rating data;
3. use the four-question template to describe a new prediction rule;
4. complete the learned state and prediction rule of an sklearn-style estimator;
5. interpret evaluation results without using test labels to redesign the model.


## Point distribution

| Question | Topic | Points |
|---|---|---:|
| 1 | Read the prepared rating data | 25 |
| 2 | Extend the mean baselines | 50 |
| 3 | Evaluate and explain | 25 |
|  | **Total** | **100** |


# 0. Setup — run these cells

The following cells load and prepare the data. They are provided for you and are not graded.


In [ ]:
import numpy as np
import pandas as pd

from IPython.display import display
from sklearn.base import BaseEstimator, RegressorMixin
from sklearn.preprocessing import LabelEncoder
from sklearn.utils.validation import check_is_fitted

pd.set_option('display.max_rows', 12)
pd.set_option('display.float_format', lambda x: f'{x:.3f}')


In [ ]:
NETFLIX_BASE_URL = (
    'https://raw.githubusercontent.com/'
    'statmlben/CUHK-STAT3009/main/dataset/netflix'
)

COLUMNS = ['user_id', 'movie_id', 'rating']
ID_TYPES = {'user_id': 'string', 'movie_id': 'string'}

train_raw = pd.read_csv(
    f'{NETFLIX_BASE_URL}/train.csv',
    usecols=COLUMNS,
    dtype=ID_TYPES,
)[COLUMNS]

test_raw = pd.read_csv(
    f'{NETFLIX_BASE_URL}/test.csv',
    usecols=COLUMNS,
    dtype=ID_TYPES,
)[COLUMNS]

# Until Question 3, use only these two test input columns.
test_inputs_raw = test_raw[['user_id', 'movie_id']].copy()

display(train_raw.head())


## Prepared `X` and `y`

We need one consistent numerical code for each user and movie across the training and test input rows. The complete preprocessing is provided below. Read it once, then run it.


In [ ]:
all_user_ids = pd.concat(
    [train_raw['user_id'], test_inputs_raw['user_id']],
    ignore_index=True,
)
all_movie_ids = pd.concat(
    [train_raw['movie_id'], test_inputs_raw['movie_id']],
    ignore_index=True,
)

user_encoder = LabelEncoder().fit(all_user_ids)
item_encoder = LabelEncoder().fit(all_movie_ids)

train_users = user_encoder.transform(train_raw['user_id'])
train_items = item_encoder.transform(train_raw['movie_id'])
test_users = user_encoder.transform(test_inputs_raw['user_id'])
test_items = item_encoder.transform(test_inputs_raw['movie_id'])

X_train = np.column_stack([train_users, train_items]).astype(int)
y_train = train_raw['rating'].to_numpy(dtype=float)
X_test = np.column_stack([test_users, test_items]).astype(int)

print('X_train:', X_train.shape)
print('y_train:', y_train.shape)
print('X_test: ', X_test.shape)


## Prepared classroom baselines

The global-, user-, and item-mean calculations repeat the lecture, so they are supplied below using the same NumPy operations as in class: `np.full`, `set(...)`, Boolean masks, and array indexing.


In [ ]:
global_mean = y_train.mean()

user_means = np.full(
    X_train[:, 0].max() + 1, global_mean
)
for user in set(X_train[:, 0]):
    ratings = y_train[X_train[:, 0] == user]
    user_means[user] = ratings.mean()

item_means = np.full(
    X_train[:, 1].max() + 1, global_mean
)
for item in set(X_train[:, 1]):
    ratings = y_train[X_train[:, 1] == item]
    item_means[item] = ratings.mean()

y_pred_global = np.full(len(X_test), global_mean)

y_pred_user = np.full(len(X_test), global_mean)
valid_users = (X_test[:, 0] >= 0) & (X_test[:, 0] < len(user_means))
y_pred_user[valid_users] = user_means[X_test[valid_users, 0]]

y_pred_item = np.full(len(X_test), global_mean)
valid_items = (X_test[:, 1] >= 0) & (X_test[:, 1] < len(item_means))
y_pred_item[valid_items] = item_means[X_test[valid_items, 1]]

print('global mean:', round(global_mean, 3))
print('prediction vectors:', y_pred_global.shape, y_pred_user.shape, y_pred_item.shape)


# Question 1 — Read the prepared rating data [25 points]

Each training row is a rating triple

$$
(u,i,r_{ui})
=
(\text{user ID},\text{movie ID},\text{rating}).
$$

The test input contains $(u,i)$ pairs. Its ratings remain hidden until Question 3.


## Q1.1 A small training-data summary [10 points]

**Your task:** replace the three blanks with pandas expressions. Use the raw training table.


In [ ]:
# YOUR TASK
n_ratings = ...
n_users = ...
n_movies = ...

training_summary = pd.Series({
    'ratings': n_ratings,
    'users': n_users,
    'movies': n_movies,
})
display(training_summary)


## Q1.2 Train and test play different roles [15 points]

**Your task:** use set subtraction to find user and movie IDs that occur in the test inputs but not in training. Replace the two blanks, run the cell, and then answer the questions below.

You should find 74 test-only users and 584 test-only movies.


In [ ]:
# YOUR TASK
test_only_users = ...
test_only_movies = ...

print('users appearing only in test inputs:', len(test_only_users))
print('movies appearing only in test inputs:', len(test_only_movies))


Answer in three or four sentences:

1. Why may we use the test **user and movie IDs** when creating a shared numerical encoding?
2. Why must the test **ratings** stay hidden while we define and fit the prediction rules?
3. What should a user-mean model do for a user that never appears in training?

**Your answer:**

> Write your answer here.


# Question 2 — Extend the mean baselines [50 points]

We will build one new rule without rewriting the classroom estimators. For a user–movie pair $(u,i)$, define

$$
f(u,i)=\frac{\mu+a_u+b_i}{3},
$$

where $\mu$ is the global training mean, $a_u$ is the user mean, and $b_i$ is the movie mean. If a user or movie was absent from training, its component uses $\mu$.


## Q2.1 Describe `ThreeMeanRS` with the four questions [15 points]

Complete each line with one or two sentences or a mathematical expression.

- **Model:**
- **Learned parameters:**
- **Hyperparameters:**
- **Loss / fitting criterion:**

For the final line, describe how each mean is fitted. You may use the fact that a mean minimizes squared error among constant predictions.


## Q2.2 Complete the learned state and prediction rule [25 points]

Complete the marked lines using the same pattern as the classroom estimators and the prepared baselines above:

1. initialize learned arrays with `np.full`;
2. use `set(...)` loops and Boolean masks to learn user and movie means;
3. begin predictions at the global fallback;
4. replace only valid user and movie positions;
5. return the equal-weight average of the three components.


In [ ]:
class ThreeMeanRS(RegressorMixin, BaseEstimator):
    def fit(self, X, y):
        X = np.asarray(X)
        y = np.asarray(y, dtype=float)
        users = X[:, 0].astype(int)
        items = X[:, 1].astype(int)

        self.global_mean_ = ...

        self.user_means_ = np.full(
            users.max() + 1, self.global_mean_
        )
        for user in set(users):
            ratings = ...
            self.user_means_[user] = ...

        self.item_means_ = np.full(
            items.max() + 1, self.global_mean_
        )
        for item in set(items):
            ratings = ...
            self.item_means_[item] = ...

        return self

    def predict(self, X):
        check_is_fitted(
            self, ['global_mean_', 'user_means_', 'item_means_']
        )
        X = np.asarray(X)
        users = X[:, 0].astype(int)
        items = X[:, 1].astype(int)

        user_part = np.full(len(users), self.global_mean_)
        valid_users = ...
        user_part[valid_users] = ...

        item_part = np.full(len(items), self.global_mean_)
        valid_items = ...
        item_part[valid_items] = ...

        return ...


Run the completed estimator. You should obtain one prediction for every row of `X_test`.


In [ ]:
three_mean_model = ThreeMeanRS().fit(X_train, y_train)
y_pred_three = three_mean_model.predict(X_test)

print('number of predictions:', len(y_pred_three))
print('first five predictions:', np.round(y_pred_three[:5], 3))


## Q2.3 Read the fitted estimator [10 points]

Answer in two or three sentences:

1. Which fitted attributes store the learned mathematical parameters?
2. Why does `predict` need a fallback even though the encoders know all test user and movie IDs?

**Your answer:**

> Write your answer here.


# Question 3 — Evaluate and explain [25 points]

All four prediction rules are now fixed. We may reveal the classroom test ratings once for final evaluation. The RMSE function is provided; you will construct the comparison table and then write one reusable report for overall, warm-start, and cold-start RMSE.


In [ ]:
def rmse(y_true, y_pred):
    return np.sqrt(np.mean((y_true - y_pred) ** 2))


y_test_demo = test_raw['rating'].to_numpy(dtype=float)

predictions = {
    'Global mean': y_pred_global,
    'User mean': y_pred_user,
    'Item mean': y_pred_item,
    'Three mean': y_pred_three,
}

# YOUR TASK: calculate one RMSE per model and build the table.
evaluation_rows = []
for model_name, y_pred in predictions.items():
    model_rmse = ...
    evaluation_rows.append(...)

evaluation_results = ...
display(evaluation_results)


## Q3.1 Build and interpret the final comparison [10 points]

Complete the code cell above. Your table should contain columns named `model` and `RMSE`, sorted from the smallest RMSE to the largest. Then answer in three or four sentences:

1. Which of the four fixed rules has the lowest test RMSE?
2. Does a lower RMSE mean that this model predicts every rating better? Explain.
3. Why should we avoid changing the model after seeing this table?

**Your answer:**

> Write your answer here.


## Q3.2 Report overall, warm-start, and cold-start RMSE [15 points]

Complete `rmse_report(...)` using the same NumPy operations used elsewhere in the homework.

- A row is **warm-start** when both its user and movie appeared in training.
- A row is **cold-start** when either its user or movie was absent from training.
- The returned DataFrame should have rows `overall`, `warm_start`, and `cold_start`, with columns `n_rows` and `RMSE`.

Use `np.isin` for the seen-ID masks and Boolean indexing when calculating each RMSE.


In [ ]:
def rmse_report(X_train, X_eval, y_true, y_pred):
    # YOUR TASK: identify warm-start and cold-start rows.
    seen_user_mask = ...
    seen_movie_mask = ...
    warm_mask = ...
    cold_mask = ...

    segment_masks = {
        'overall': np.ones(len(X_eval), dtype=bool),
        'warm_start': warm_mask,
        'cold_start': cold_mask,
    }

    report_rows = []
    for segment_name, mask in segment_masks.items():
        segment_rmse = ...
        report_rows.append(...)

    return ...


segment_report = rmse_report(
    X_train, X_test, y_test_demo, y_pred_three
)
display(segment_report)


In two or three sentences, compare the overall, warm-start, and cold-start RMSE. Why can the overall RMSE hide weaker performance on cold-start rows?

**Your answer:**

> Write your answer here.


# Submission checklist

Before submitting:

- [ ] replace every `...` in the five graded code cells;
- [ ] complete all written responses;
- [ ] run the notebook from top to bottom without an error;
- [ ] keep the cell outputs visible;
- [ ] do not use `test_raw["rating"]` before Question 3.
